## Parameter-Efficient Fine-Tuning (PEFT) with LoRA

This notebook demonstrates how to fine-tune a language model using LoRA.

In [1]:
# Install necessary libraries
!pip install -qqq peft transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
print('Reinstalling torch and torchvision to resolve potential compatibility issues.')
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print('Installation complete. Please restart the Colab runtime (Runtime -> Restart runtime) before proceeding.')

Reinstalling torch and torchvision to resolve potential compatibility issues.
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

### 1. Load Pre-trained Model and Tokenizer

We will use the `bigscience/bloomz-560m` model for this demonstration.

In [ ]:
model_id = "bigscience/bloomz-560m"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Add a pad token if it doesn't exist, this is common for causal LMs
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

# Load model
foundation_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)

# Resize token embeddings to account for the added pad token
foundation_model.resize_token_embeddings(len(tokenizer))


### 2. Load and Preprocess Dataset

We'll use a 10% sample of the `Abirate/english_quotes` dataset.

In [ ]:
dataset = load_dataset("Abirate/english_quotes")

# Sample 10% of the training data
train_dataset = dataset["train"].shuffle(seed=42).select(range(int(len(dataset["train"]) * 0.1)))

def tokenize_function(examples):
    return tokenizer(examples["quote"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

# Format columns for training
tokenized_dataset = tokenized_dataset.remove_columns(["quote", "author"])
tokenized_dataset = tokenized_dataset.rename_column("input_ids", "labels") # For causal LMs, input_ids are often used as labels
tokenized_dataset.set_format("torch")

print(f"Sample tokenized entry: {tokenized_dataset[0]}")

### 3. Configure LoRA

We set up `LoraConfig` for Parameter-Efficient Fine-Tuning.

In [ ]:
lora_config = LoraConfig(
    r=8, # Rank of the update matrices. Set to a low value for smaller updates.
    lora_alpha=32, # LoRA alpha parameter. Scaling factor for LoRA updates.
    target_modules=["query_key_value"], # Specific modules to apply LoRA to.
    lora_dropout=0.05, # Dropout probability for LoRA layers.
    bias="none", # Type of bias to use, "none" is typical for LoRA.
    task_type=TaskType.CAUSAL_LM # Task type, e.g., Causal Language Modeling.
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

### 4. Setup Training Arguments and Trainer

Define training parameters and initialize the `Trainer`.

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_finetuned_model", # Directory to save checkpoints and logs
    per_device_train_batch_size=4, # Batch size per GPU/CPU for training
    gradient_accumulation_steps=2, # Number of updates steps to accumulate before performing a backward/update pass
    warmup_steps=100, # Number of warmup steps for learning rate scheduler
    max_steps=200, # Total number of training steps
    learning_rate=2e-4, # Initial learning rate
    fp16=True, # Enable mixed precision training
    logging_dir="./logs", # Directory for storing logs
    logging_steps=10, # Log every N steps
    save_strategy="steps", # Save checkpoints every N steps
    save_steps=50, # Save checkpoint every N steps
    optim="adamw_torch" # Optimizer to use
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer # Pass the tokenizer to the trainer
)

### 5. Train the Model

Start the fine-tuning process.

In [ ]:
trainer.train()

### 6. Save the Fine-tuned LoRA Model

Save only the LoRA adapter weights.

In [ ]:
peft_model.save_pretrained("./final_lora_model")

### 7. Load Saved Model for Inference

Load the base model and then attach the fine-tuned LoRA adapter.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16)
loaded_peft_model = PeftModel.from_pretrained(base_model, "./final_lora_model")

print("LoRA model loaded successfully for inference.")

### 8. Generate Text with the Fine-tuned Model

Test the model's ability to generate text based on a prompt.

In [ ]:
prompt = "The early bird catches the"
inputs = tokenizer(prompt, return_tensors="pt")

# Ensure the model is in evaluation mode
loaded_peft_model.eval()

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=20,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id # Use EOS token as pad token
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated Text: {generated_text}")